# VM.AI Parser Training

**Enable GPU first:**
- Runtime → Change runtime type → GPU (T4) → Save

Uses **uv** for fast package installation.

In [ ]:
# Run src/parser/package_colab.py, it will package in colab.zip all files needed for this notebook
!unzip -o colab.zip

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!python train.py --mode both

In [ ]:
# Save to Drive
!mkdir -p /content/drive/MyDrive/vmai/models
!cp -r /content/models/finetuned_parser /content/drive/MyDrive/vmai/models/
print('Saved to Google Drive')

## Test Model (Chat Interface)

In [ ]:
import os, shutil, re, zipfile
from google.colab import files as colab_files

model_dir = "/content/models/finetuned_parser"
if not os.path.exists(model_dir):
    print(f"Error: {model_dir} not found")
else:
    # Find latest checkpoint (optional, include for training continuity)
    checkpoints = []
    for d in os.listdir(model_dir):
        m = re.match(r"checkpoint-(\d+)", d)
        if m:
            checkpoints.append((int(m.group(1)), d))

    latest_ckpt = max(checkpoints, key=lambda x: x[0])[1] if checkpoints else None
    if latest_ckpt:
        print(f"Latest checkpoint: {latest_ckpt}")

    zip_name = "finetuned_parser.zip"
    with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
        for item in os.listdir(model_dir):
            if item.startswith(".cache"):
                continue
            # Skip older checkpoints to keep zip small
            if item.startswith("checkpoint-") and item != latest_ckpt:
                continue
            item_path = os.path.join(model_dir, item)
            if os.path.isfile(item_path):
                zf.write(item_path, item)
            else:
                for root, dirs, fnames in os.walk(item_path):
                    for f in fnames:
                        fp = os.path.join(root, f)
                        zf.write(fp, os.path.relpath(fp, model_dir))

    size_mb = os.path.getsize(zip_name) / (1024 * 1024)
    print(f"Zipped: {zip_name} ({size_mb:.1f} MB)")
    colab_files.download(zip_name)
    print("Download started!")

In [ ]:
from transformers import AutoTokenizer, T5ForConditionalGeneration
import torch
import json

model_path = '/content/models/finetuned_parser'
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = T5ForConditionalGeneration.from_pretrained(model_path)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
print('Model loaded')

In [ ]:
def predict(input_text):
    inputs = tokenizer(input_text, return_tensors='pt', truncation=True, padding='max_length', max_length=256).to(device)
    outputs = model.generate(inputs['input_ids'], max_new_tokens=128)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

test_cases = [
    'add: gym at 6am',
    'add: meditate every morning',
    'add: finish report by Friday',
    'add: team meeting at 3pm',
]

for inp in test_cases:
    output = predict(inp)
    print(f'Input:  {inp}')
    print(f'Output: {output}')
    print()

In [ ]:
print('VM.AI Parser - Interactive Test')
print('Type your prompts (or "quit" to exit)')
print()

while True:
    user_input = input('Enter: ')
    if user_input.lower() == 'quit':
        break

    if not user_input.startswith('add:') and not user_input.startswith('modify:'):
        user_input = 'add: ' + user_input

    output = predict(user_input)
    print(f'Output: {output}')
    print()